# Arriendos Gran Concepción — Portal Inmobiliario

Descarga, revisión de calidad y procesamiento de avisos de arriendo (casas y departamentos).

## 1. Setup

In [ ]:
import sys
sys.executable

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import plotly.express as px
import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import COMMUNES, TYPES
from src.portal.crawler import run_crawl, RAW_DIR
from src.geocoding import update_cache
from src.uf import get_uf

PROCESSED_DIR = ROOT / "data" / "processed"

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

## 2. Descarga
Elegir comunas y número máximo de páginas. Las búsquedas ya descargadas hoy se saltan.
**Si solo quieres analizar datos ya descargados, sáltate esta sección.**

In [ ]:
# Test run
# communes_to_run = {k: COMMUNES[k] for k in ["Concepción", "Talcahuano"]}
# MAX_PAGES = 3

# Full run
communes_to_run = COMMUNES
MAX_PAGES = 50

In [ ]:
with ThreadPoolExecutor(max_workers=1) as executor:
    run_dir = executor.submit(run_crawl, communes=communes_to_run, max_pages=MAX_PAGES).result()

print("Guardados en:", run_dir)

## 3. Carga

In [ ]:
# Latest run, or set a specific date: RUN = "2026-10-24"
RUN = sorted(p.name for p in RAW_DIR.iterdir() if p.is_dir())[-1]
run_dir = RAW_DIR / RUN

files = sorted(run_dir.glob("*.csv"))
df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

print(f"Run {RUN}: {len(files)} files, {len(df)} listings")

In [ ]:
df.pivot_table(index="commune", columns="property_type",
               values="id", aggfunc="count", fill_value=0)

## 4. Calidad

In [ ]:
quality = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(1),
})
quality[quality["missing"] > 0].sort_values("missing_%", ascending=False)

## 5. Procesamiento
1. Unir y quitar duplicados
2. Filtrar avisos inválidos
3. Agregar coordenadas (solo se consultan direcciones nuevas; el resto sale del caché)
4. Guardar en `data/processed/`

In [ ]:
# 5.1  Unir, filtrar y convertir UF a pesos
MAX_UF_RENT = 300      # above this it's a sale price (~$12M per month)
MAX_BEDROOMS = 10      # above this it's not a dwelling (offices, hostel, institute)

consolidated = (
    df.drop_duplicates(subset=["id"])
      .drop(columns=["total_m2", "broker"], errors="ignore")
      .dropna(subset=["price"])
)

# Filter invalid listings
is_sale_price = (consolidated["currency"] == "UF") & (consolidated["price"] > MAX_UF_RENT)
not_dwelling = consolidated["bedrooms"] > MAX_BEDROOMS
excluded = consolidated[is_sale_price | not_dwelling]
consolidated = consolidated[~(is_sale_price | not_dwelling)].copy()

print(f"Rows: {len(df)} -> {len(consolidated)} ({len(excluded)} excluded)")
display(excluded[["property_type", "commune", "price", "currency", "bedrooms", "title"]])

# Convert UF to CLP using the UF value of each download date
uf_by_date = {d: get_uf(d) for d in consolidated["scraped_at"].unique()}
consolidated["uf_value"] = consolidated["scraped_at"].map(uf_by_date)
consolidated["price_clp"] = (
    consolidated["price"]
    .where(consolidated["currency"] == "CLP",
           consolidated["price"] * consolidated["uf_value"])
    .round(0)
)

In [ ]:
# 5.3a  PRUEBA: geolocalizar solo 30 direcciones (ejecutar una vez y revisar)
cache = update_cache(consolidated.head(30))
cache[["location", "precision", "lat", "lon"]].tail(10)

In [ ]:
## borra el cache de la geolocalización para que se vuelva a intentar la geolocalización de las direcciones que no se encontraron
##from src.geocoding import forget_not_found
##forget_not_found()

In [ ]:
# 5.3b  Geolocalizar todo (la primera vez tarda 20-40 min)
cache = update_cache(consolidated)
cache["precision"].value_counts()

In [ ]:
# 5.4  Unir coordenadas y guardar
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

processed = consolidated.merge(
    cache[["location", "lat", "lon", "precision","matched_address"]],
    on="location", how="left",
)

output_file = PROCESSED_DIR / f"portal_{RUN}.csv"
processed.to_csv(output_file, index=False, encoding="utf-8")

print(f"{len(processed)} listings, {processed['lat'].notna().sum()} with coordinates")
print("Saved:", output_file.relative_to(ROOT))

In [ ]:
# 5.5  Distribución de precios en pesos (verificación)
processed.groupby("property_type")["price_clp"].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
).round(0)

In [ ]:
# 5.6  Coordenadas por comuna y precisión
summary = (
    processed.assign(precision=processed["precision"].fillna("not_found"))
    .pivot_table(index="commune", columns="precision",
                 values="id", aggfunc="count", fill_value=0)
)
summary["%_with_coords"] = (
    100 * (1 - summary.get("not_found", 0) / summary.sum(axis=1))
).round(1)
summary

In [ ]:
# 5.7  Direcciones sin coordenadas
missing = processed[processed["lat"].isna()]
missing[["commune", "location", "neighborhood"]].drop_duplicates().head(20)

## 6. Mapa de verificación
Solo para comprobar que los puntos caen donde deben.

In [ ]:
points = processed.dropna(subset=["lat"]).copy()

# Readable text for the tooltip
points["price_text"] = points.apply(
    lambda r: f"${r['price_clp']:,.0f}".replace(",", ".")
    + (f"  (UF {r['price']:,.1f})".replace(".", ",") if r["currency"] == "UF" else ""),
    axis=1,
)
points["m2_text"] = points["usable_m2"].map(lambda v: f"{v:.0f} m²" if pd.notna(v) else "sin dato")
points["bed_text"] = points["bedrooms"].map(lambda v: f"{v:.0f} dorm." if pd.notna(v) else "")
points["bath_text"] = points["bathrooms"].map(lambda v: f"{v:.0f} baños" if pd.notna(v) else "")
points["neighborhood_text"] = points["neighborhood"].fillna("")

fig = px.scatter_map(
    points, lat="lat", lon="lon", color="commune",
    custom_data=["title", "price_text", "property_type", "bed_text", "bath_text",
                 "m2_text", "neighborhood_text", "commune"],
    zoom=10, height=650,
    labels={"commune": "Comuna"},
)

fig.update_traces(
    marker=dict(size=6, opacity=0.75),
    hovertemplate=(
        "<b>%{customdata[1]}</b><br>"
        "%{customdata[0]}<br>"
        "<span style='color:#666'>%{customdata[2]} · %{customdata[3]} · "
        "%{customdata[4]} · %{customdata[5]}</span><br>"
        "<span style='color:#666'>%{customdata[6]}, %{customdata[7]}</span>"
        "<extra></extra>"
    ),
)

fig.update_layout(
    hoverlabel=dict(bgcolor="white", font_size=13, font_family="Arial",
                    bordercolor="#ccc"),
    margin=dict(l=0, r=0, t=0, b=0),
    legend=dict(bgcolor="rgba(255,255,255,0.85)", x=0.01, y=0.99),
)
fig.show()

In [ ]:

pts = processed.dropna(subset=["lat"]).copy()

# Center of each commune = median of its points (robust to a few wrong ones)
centers = pts.groupby("commune")[["lat", "lon"]].median()
pts = pts.join(centers, on="commune", rsuffix="_center")

# Approximate distance in km (good enough at this scale)
pts["km_from_center"] = np.sqrt(
    ((pts["lat"] - pts["lat_center"]) * 111) ** 2
    + ((pts["lon"] - pts["lon_center"]) * 111 * np.cos(np.radians(pts["lat"]))) ** 2
).round(1)

(pts.sort_values("km_from_center", ascending=False)
    [["commune", "location", "precision", "km_from_center", "lat", "lon"]]
    .head(20))

In [ ]:
processed.groupby(["property_type", "currency"])["price"].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
).round(1)

In [ ]:
suspicious = processed[
    ((processed["currency"] == "UF") & (processed["price"] > 100))
    | ((processed["currency"] == "CLP") & (processed["price"] > 4_000_000))
    | ((processed["currency"] == "CLP") & (processed["price"] < 250_000))
]

(suspicious[["property_type", "commune", "price", "currency",
             "bedrooms", "usable_m2", "title", "url"]]
    .sort_values(["currency", "price"]))